## 실습 1. Chapter 01 전처리 데이터 불러오기

In [1]:
import pandas as pd

DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


### 실행 결과 요약

- 데이터 크기 확인: 총 **199행, 8개 컬럼**으로 구성되어 있다.
- 컬럼 확인: `순위`, `판매상품ID`, `상품명`, `판매가`, `저자`, `출판사`, `발행일`, `분야` 컬럼이 정상적으로 존재한다.
- 결측치 확인: `상품명` 컬럼의 결측치는 **0개**로 확인되었다.
- 제목 확인: `소년이 온다`, `모순`, `결국 국민이 합니다` 등 앞의 10개 상품명이 정상적으로 출력되었다.
- 결과 확인: 한글이 깨지지 않고 도서 제목이 정상적으로 표시되어 이후 텍스트 분석에 사용할 수 있는 상태임을 확인했다.

## 실습 2. 분석할 제목 문자열 준비하기

In [2]:
# 상품명 컬럼을 문자열 형태로 준비
titles = (
    df_books["상품명"]
    .fillna("")      # 결측값이 있으면 빈 문자열로 변경
    .astype(str)     # 모든 값을 문자열로 변환
    .str.strip()     # 앞뒤 공백 제거
)

# 빈 문자열 제거 후 인덱스 다시 정리
titles = titles[titles != ""].reset_index(drop=True)

# 사용할 제목 개수 확인
print("사용할 제목 수:", len(titles))

# 앞에서 10개 제목 확인
titles.head(10)

사용할 제목 수: 199


0                      소년이 온다
1                          모순
2                  결국 국민이 합니다
3                         혼모노
4                          급류
5                    초역 부처의 말
6               청춘의 독서(특별증보판)
7                어른의 행복은 조용하다
8                       채식주의자
9    단 한 번의 삶(강물에디션 활판인쇄 한정판)
Name: 상품명, dtype: str

### 실행 결과 요약

- 문자열 준비: `상품명` 컬럼의 도서 제목이 문자열 형태로 정상적으로 준비되었다.
- 제목 확인: 앞의 10개 제목을 확인한 결과 한글이 깨지지 않고 정상적으로 출력되었다.
- 원본 유지 확인: `소년이 온다`, `모순`, `결국 국민이 합니다` 등 실제 상품명이 그대로 유지되었다.
- 결과 확인: 현재 `titles` 데이터는 Vectorizer에 전달할 수 있는 형태로 준비되었다.

## 실습 3. 머신러닝은 왜 텍스트를 숫자로 바꿀까?

- 머신러닝 알고리즘은 일반적으로 **숫자 형태의 데이터**를 입력받아 계산한다.
- `[1, 0, 2, 0]`, `[0, 1, 0, 3]`과 같은 숫자 벡터는 계산할 수 있지만, `"데이터 분석을 위한 파이썬"` 같은 문자열을 그대로 학습에 사용할 수는 없다.
- 따라서 도서 제목에 어떤 단어가 사용되었는지 확인하고, 단어마다 열(column)을 만든 뒤 등장 여부나 등장 횟수를 숫자로 기록한다.
- 이 과정을 통해 텍스트를 머신러닝이 처리할 수 있는 **숫자 벡터**로 변환한다.
- 이러한 과정을 **텍스트 벡터화(Vectorization)**라고 한다.

```text
도서 제목
    ↓
사용된 단어 확인
    ↓
단어마다 열(column) 생성
    ↓
등장 여부 또는 등장 횟수를 숫자로 기록
    ↓
숫자 벡터


## 실습4. 아주 작은 예제로 먼저 이해하기

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. 예시 문장
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

# 2. CountVectorizer 생성
vectorizer = CountVectorizer()

# 3. 문장을 숫자 벡터로 변환
X = vectorizer.fit_transform(sample_docs)4. 

# 4. 단어 목록 확인
print("단어 순서:")
print(vectorizer.get_feature_names_out())

# 5. 숫자 벡터 확인
print("\n벡터 결과:")
print(X.toarray())

단어 순서:
['데이터' '머신러닝' '분석' '입문' '파이썬']

벡터 결과:
[[1 0 1 0 1]
 [0 1 0 0 1]
 [1 0 1 1 0]]


### 실행 결과 요약

- 단어 순서 확인: `데이터`, `머신러닝`, `분석`, `입문`, `파이썬` 순서로 총 **5개의 단어**가 열로 구성되었다.
- 벡터 변환 확인: 각 문장이 단어 등장 횟수를 기준으로 숫자 벡터로 변환되었다.
- 첫 번째 문장 확인: `파이썬 데이터 분석`은 `[1, 0, 1, 0, 1]`로 표현되었다.
- 두 번째 문장 확인: `파이썬 머신러닝`은 `[0, 1, 0, 0, 1]`로 표현되었다.
- 세 번째 문장 확인: `데이터 분석 입문`은 `[1, 0, 1, 1, 0]`로 표현되었다.
- 결과 확인: 각 숫자는 해당 단어가 문장에 몇 번 등장했는지를 나타내며, 텍스트가 머신러닝에서 사용할 수 있는 숫자 형태로 변환된 것을 확인했다.

## 실습5. Bag of Words 이해하기

### Bag of Words(BoW) 개념

- Bag of Words는 문장을 단어들의 모음으로 보고, 각 단어가 몇 번 등장했는지를 기준으로 표현하는 방식이다.
- 단어의 순서보다는 어떤 단어가 얼마나 등장했는지에 초점을 둔다.
- 따라서 `파이썬 데이터 분석`과 `데이터 파이썬 분석`처럼 단어 순서만 다르고 등장 단어와 횟수가 같다면 동일한 Count 벡터가 될 수 있다.
- 장점: 구조가 단순하고 이해하기 쉬우며 기본적인 텍스트 분석에 활용하기 좋다.
- 한계: 단어 순서와 문맥 정보를 충분히 표현하지 못한다.
- 결과 확인: 앞에서 확인한 CountVectorizer의 동작 방식이 Bag of Words 개념에 해당함을 이해했다.

## 실습6. CounterVectorizer 설치 확인하기

CountVectorizer는 scikit-learn에 포함되어 있습니다.

 

설치가 필요한 경우 다음과 같이 실행합니다.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

### 실행 결과 요약

- 라이브러리 확인: `CountVectorizer`는 `scikit-learn`에 포함되어 있다.
- import 확인: `from sklearn.feature_extraction.text import CountVectorizer`가 오류 없이 실행되었다.
- 결과 확인: 현재 Notebook에서 CountVectorizer를 사용할 준비가 완료되었다.

## 실습7. 작은 예제에 CountVectorizer 적용하기

In [6]:
# CountVectorizer 불러오기
from sklearn.feature_extraction.text import CountVectorizer

# 예시 문장
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문"
]

# CountVectorizer 객체 생성
count_vectorizer = CountVectorizer()

# 단어를 학습하고 문장을 숫자 벡터로 변환
X_count_sample = count_vectorizer.fit_transform(sample_docs)

# 1. CountVectorizer가 만든 수치 행렬 확인
print("수치 행렬:")
print(X_count_sample)

# 2. 일반적인 2차원 배열로 변환
print("\n2차원 배열:")
print(X_count_sample.toarray())

# 단어 순서 확인
print("\n단어 순서:")
print(count_vectorizer.get_feature_names_out())

수치 행렬:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 8 stored elements and shape (3, 5)>
  Coords	Values
  (0, 4)	1
  (0, 0)	1
  (0, 2)	1
  (1, 4)	1
  (1, 1)	1
  (2, 0)	1
  (2, 2)	1
  (2, 3)	1

2차원 배열:
[[1 0 1 0 1]
 [0 1 0 0 1]
 [1 0 1 1 0]]

단어 순서:
['데이터' '머신러닝' '분석' '입문' '파이썬']


### 실행 결과 요약

- 벡터 변환: `CountVectorizer`를 사용해 각 문장을 숫자 형태의 행렬로 변환했다.
- 수치 행렬 확인: 변환 결과는 내부적으로 희소 행렬(Sparse Matrix) 형태로 저장되었다.
- 2차원 변환: `toarray()`를 사용해 결과를 일반적인 2차원 숫자 배열로 변환했다.
- 행과 열 확인: 각 행은 하나의 문장을 의미하고, 각 열은 `데이터`, `머신러닝`, `분석`, `입문`, `파이썬`과 같은 단어를 의미한다.
- 결과 확인: 각 숫자는 해당 문장에서 각 단어가 등장한 횟수를 나타냄을 확인했다.

## 실습8. 생성된 단어 사전 확인하기

Vectorizer가 어떤 단어를 열로 만들었는지 확인합니다.

In [7]:
# CountVectorizer가 만든 단어 사전 확인
feature_names = count_vectorizer.get_feature_names_out()

# 단어 순서 출력
print("단어 사전:")
print(feature_names)

단어 사전:
['데이터' '머신러닝' '분석' '입문' '파이썬']


### 실행 결과 요약

- 단어 사전 확인: `get_feature_names_out()`을 사용해 CountVectorizer가 만든 단어 목록을 확인했다.
- 단어 순서 확인: `데이터`, `머신러닝`, `분석`, `입문`, `파이썬` 순서로 열이 구성되었다.
- 벡터 해석 기준 확인: 숫자 벡터의 각 위치는 이 단어 순서와 대응한다.
- 결과 확인: Count 벡터를 해석할 때는 반드시 `feature_names`의 순서와 함께 확인해야 함을 알 수 있다.

## 실습9. 단어-문서 행렬 확인하기

In [8]:
# pandas 불러오기
import pandas as pd

# CountVectorizer 결과를 2차원 배열로 변환
count_array = X_count_sample.toarray()

# 문장을 행, 단어를 열로 가지는 DataFrame 생성
sample_count_df = pd.DataFrame(
    count_array,
    columns=feature_names,
    index=sample_docs
)

# 전체 표 확인
sample_count_df

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


### 실행 결과 요약

- 배열 변환: `toarray()`를 사용해 CountVectorizer 결과를 일반적인 2차원 배열로 변환했다.
- DataFrame 생성: 각 문장을 행으로, 각 단어를 열로 가지는 표를 만들었다.
- 값 의미 확인: 각 값은 해당 문장에서 해당 단어가 등장한 횟수를 나타낸다.
- 첫 번째 문장 확인: `파이썬 데이터 분석` 행에서 `파이썬`, `데이터`, `분석`은 각각 1, `머신러닝`, `입문`은 각각 0으로 나타났다.
- 결과 확인: 행은 문서, 열은 단어, 값은 단어 등장 횟수라는 CountVectorizer의 구조를 확인했다.

## 실습10. 같은 단어가 여러 번 나오면 어떻게 될까?

In [9]:
# 같은 단어가 여러 번 등장하는 예제 문장
repeat_docs = [
    "파이썬 파이썬 데이터",
    "데이터 분석"
]

# CountVectorizer 객체 생성
repeat_vectorizer = CountVectorizer()

# 단어를 학습하고 문장을 숫자 벡터로 변환
X_repeat = repeat_vectorizer.fit_transform(repeat_docs)

# 결과를 보기 쉬운 DataFrame으로 변환
repeat_df = pd.DataFrame(
    X_repeat.toarray(),
    columns=repeat_vectorizer.get_feature_names_out(),
    index=repeat_docs
)

# 결과 확인
repeat_df

,데이터,분석,파이썬
파이썬 파이썬 데이터,1,0,2
데이터 분석,1,1,0


### 실행 결과 요약

- 예제 문장 준비: 같은 단어가 반복되는 문장을 포함해 CountVectorizer를 적용했다.
- 벡터 변환 확인: 각 문장에서 단어가 등장한 횟수가 숫자로 기록되었다.
- 반복 횟수 확인: `파이썬 파이썬 데이터` 문장에서 `파이썬`이 두 번 등장해 값이 **2**로 나타났다.
- 값 의미 확인: CountVectorizer는 단어의 단순 존재 여부가 아니라 기본적으로 **등장 횟수**를 기록한다.
- 결과 확인: 같은 단어가 여러 번 등장하면 해당 열의 값도 등장 횟수만큼 증가함을 확인했다.

## 실습 11. CountVectorizer의 기본 토큰 기준 확인하기

 

CountVectorizer()는 기본 설정에서 문자열을 내부적으로 나누어 토큰을 만듭니다.

 

기본 토큰 패턴은 일반적으로 두 글자 이상의 단어 문자를 대상으로 합니다.

 

따라서 한 글자 토큰은 기본 설정에서 제외될 수 있습니다.

 

예를 들어 다음 결과를 직접 확인해 봅니다.

In [10]:
# CountVectorizer의 기본 토큰 기준 확인용 예제
test_docs = [
    "AI 데이터 분석 R 파이썬"
]

# CountVectorizer 객체 생성
test_vectorizer = CountVectorizer()

# 문장을 학습하고 숫자 벡터로 변환
X_test = test_vectorizer.fit_transform(test_docs)

# 어떤 단어가 토큰으로 선택되었는지 확인
print("추출된 단어:")
print(test_vectorizer.get_feature_names_out())

추출된 단어:
['ai' '데이터' '분석' '파이썬']


### 실행 결과 요약

- 예제 문장 준비: `AI 데이터 분석 R 파이썬` 문장에 CountVectorizer를 적용했다.
- 토큰 기준 확인: 기본 설정에서는 일반적으로 두 글자 이상의 단어 문자가 토큰으로 사용된다.
- 결과 확인: `AI`, `데이터`, `분석`, `파이썬`은 토큰으로 추출되고, 한 글자인 `R`은 제외될 수 있다.
- 의미 확인: CountVectorizer의 결과는 어떤 문자열을 단어로 인식하는지에 따라 달라질 수 있다.
- 결과 확인: 기본 토큰 기준을 그대로 사용하기보다 분석 목적에 맞는지 직접 확인하는 것이 중요함을 알 수 있다.

## 실습 12. 실제 도서 제목에 CountVectorizer 적용하기

 

이제 실제 상품명 데이터에 적용합니다.

In [11]:
# 실제 도서 제목에 CountVectorizer 적용
count_vectorizer = CountVectorizer()

# 제목을 숫자 벡터로 변환
X_count = count_vectorizer.fit_transform(titles)

# CountVectorizer가 만든 단어 목록
count_terms = count_vectorizer.get_feature_names_out()

# 행렬 크기 확인
print("문서 수:", X_count.shape[0])
print("단어 수:", X_count.shape[1])
print("행렬 크기:", X_count.shape)

문서 수: 199
단어 수: 536
행렬 크기: (199, 536)


### 실행 결과 요약

- 문서 수 확인: CountVectorizer에 사용된 도서 제목은 총 **199개**이다.
- 단어 수 확인: CountVectorizer가 생성한 단어는 총 **536개**이다.
- 행렬 크기 확인: 변환된 행렬의 크기는 **(199, 536)**이다.
- 구조 확인: 행은 각각의 도서 제목을 의미하고, 열은 CountVectorizer가 만든 단어를 의미한다.
- 결과 확인: 199개의 도서 제목이 536개의 단어 기준으로 숫자 벡터 형태로 변환되었다.

## 실습 13. vocabulary_ 확인하기

 

Vectorizer 내부에는 단어와 열 번호의 대응 정보가 있습니다.

In [12]:
# vocabulary_에서 단어와 열 번호 확인
vocab_items = list(count_vectorizer.vocabulary_.items())

# 앞의 20개만 출력
print(vocab_items[:20])

[('소년이', 278), ('온다', 355), ('모순', 194), ('결국', 67), ('국민이', 84), ('합니다', 511), ('혼모노', 524), ('급류', 92), ('초역', 460), ('부처의', 236), ('청춘의', 458), ('독서', 150), ('특별증보판', 485), ('어른의', 332), ('행복은', 517), ('조용하다', 432), ('채식주의자', 454), ('번의', 220), ('강물에디션', 57), ('활판인쇄', 528)]


### 실행 결과 요약

- vocabulary 확인: 실제 도서 제목에서 추출된 단어와 각 단어의 열 번호를 확인했다.
- 예시 확인: `소년이`는 278번 열, `온다`는 355번 열, `모순`은 194번 열에 위치했다.
- 추가 확인: `혼모노`, `급류`, `채식주의자`, `강물에디션`, `활판인쇄` 등 실제 제목에 포함된 단어들이 vocabulary에 등록되어 있었다.
- 열 번호 의미 확인: 각 단어 옆의 숫자는 단어 빈도가 아니라 Count 행렬에서 해당 단어가 위치한 **열 번호**이다.
- 결과 확인: CountVectorizer가 실제 도서 제목에서 단어 사전을 만들고 각 단어에 고유한 열 위치를 부여했음을 확인했다.

## 실습 14. 희소 행렬(Sparse Matrix) 이해하기

 

실제 도서 제목 전체를 벡터화하면 많은 값이 0이 됩니다.

 

예를 들어 특정 도서 제목에 파이썬이라는 단어가 없다면 그 열의 값은 0입니다.

 

수백 또는 수천 개 단어를 만들면 대부분의 문서에서 대부분의 단어는 등장하지 않습니다.

In [13]:
# X_count의 자료형 확인
print("행렬 타입:")
print(type(X_count))

# 전체 원소 개수 확인
total_values = X_count.shape[0] * X_count.shape[1]

# 0이 아닌 값의 개수 확인
nonzero_values = X_count.nnz

# 0인 값의 개수 계산
zero_values = total_values - nonzero_values

print("\n전체 원소 수:", total_values)
print("0이 아닌 값 수:", nonzero_values)
print("0인 값 수:", zero_values)

# 0의 비율 확인
zero_ratio = zero_values / total_values * 100
print(f"0의 비율: {zero_ratio:.2f}%")

행렬 타입:
<class 'scipy.sparse._csr.csr_matrix'>

전체 원소 수: 106664
0이 아닌 값 수: 684
0인 값 수: 105980
0의 비율: 99.36%


In [14]:
# 앞의 10개 도서 × 앞의 20개 단어만 선택
matrix_sample = X_count[:10, :20].toarray()

# 보기 쉽게 DataFrame으로 변환
matrix_df = pd.DataFrame(
    matrix_sample,
    columns=count_terms[:20],
    index=titles[:10]
)

matrix_df

,100,100만,100일,10만,10일,10주년,110,13,14,19,1984,1차,2025,2026,20만,20주년,28시간에,29,30만,30만부
상품명,,,,,,,,,,,,,,,,,,,,
소년이 온다,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
모순,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
결국 국민이 합니다,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
혼모노,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
급류,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
초역 부처의 말,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
청춘의 독서(특별증보판),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
어른의 행복은 조용하다,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
채식주의자,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 실행 결과 요약

- 행렬 타입 확인: `X_count`는 `scipy.sparse._csr.csr_matrix` 형태의 **희소 행렬**로 저장되어 있다.
- 전체 원소 수 확인: 행렬 전체에는 총 **106,664개**의 값이 들어갈 수 있다.
- 0이 아닌 값 확인: 실제로 값이 존재하는 위치는 **684개**이다.
- 0인 값 확인: 나머지 **105,980개**는 0으로 구성되어 있다.
- 희소성 확인: 전체 값 중 **99.36%가 0**으로 나타났다.
- 결과 확인: 실제 도서 제목 벡터는 대부분의 값이 0이므로, 전체를 일반 배열로 저장하기보다 희소 행렬로 저장하는 것이 메모리 효율적임을 확인했다.

### 실행 결과 요약

- 행렬 타입 확인: `X_count`는 `scipy.sparse._csr.csr_matrix` 형태의 **희소 행렬**로 저장되어 있다.
- 전체 원소 수 확인: 행렬 전체에는 총 **106,664개**의 값이 들어갈 수 있다.
- 0이 아닌 값 확인: 실제로 값이 존재하는 위치는 **684개**이다.
- 0인 값 확인: 나머지 **105,980개**는 0으로 구성되어 있다.
- 희소성 확인: 전체 값 중 **99.36%가 0**으로 나타났다.
- 매트릭스 확인: 전체 행렬 중 일부를 선택해 일반적인 2차원 표 형태로 확인했다.
- 행 확인: 각 행은 하나의 도서 제목을 의미하고, 각 열은 CountVectorizer가 만든 단어를 의미한다.
- 값 확인: 각 셀의 값은 해당 도서 제목에서 해당 단어가 등장한 횟수를 나타낸다.
- 결과 확인: 실제 도서 제목 벡터는 대부분의 값이 0으로 구성되어 있으며, 일부 매트릭스를 직접 확인해 희소 행렬의 구조를 이해할 수 있었다.
- 메모리 효율 확인: 전체 행렬을 일반 배열로 변환하기보다 희소 행렬 상태로 유지하고 필요한 일부만 확인하는 것이 더 효율적임을 확인했다.

## 실습 15. 실제 데이터의 첫 번째 제목 벡터 확인하기

In [15]:
# 첫 번째 도서 제목 확인
print("첫 번째 제목:")
print(titles.iloc[0])

# 첫 번째 제목에 해당하는 벡터 가져오기
first_row = X_count.getrow(0)

# 값이 0이 아닌 단어의 열 번호와 등장 횟수 확인
indices = first_row.indices
values = first_row.data

# 열 번호를 실제 단어와 연결
first_title_terms = [
    (count_terms[index], value)
    for index, value in zip(indices, values)
]

# 결과 출력
print("\n첫 번째 제목의 단어와 등장 횟수:")
print(first_title_terms)

첫 번째 제목:
소년이 온다

첫 번째 제목의 단어와 등장 횟수:
[('소년이', np.int64(1)), ('온다', np.int64(1))]


### 실행 결과 요약

- 첫 번째 제목 확인: 첫 번째 도서 제목은 `소년이 온다`로 확인되었다.
- 단어 추출 확인: CountVectorizer는 제목에서 `소년이`, `온다` 두 단어를 추출했다.
- 등장 횟수 확인: `소년이`와 `온다`는 각각 **1번**씩 등장했다.
- 원본 비교: 추출된 두 단어가 실제 제목에 포함된 단어와 일치했다.
- 결과 확인: 첫 번째 제목이 CountVectorizer를 통해 올바른 단어와 등장 횟수로 표현되었음을 확인했다.

## 실습 16. 전체 데이터에서 많이 등장한 단어 확인하기

 

Count 행렬의 각 열을 합하면 전체 문서에서 각 단어가 등장한 총 횟수를 계산할 수 있습니다.

In [16]:
# numpy 불러오기
import numpy as np

# 각 단어가 전체 제목에서 등장한 횟수 합계 계산
count_sums = np.asarray(X_count.sum(axis=0)).ravel()

# 단어와 전체 등장 횟수를 DataFrame으로 정리
count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums
})

# 등장 횟수가 많은 순서대로 정렬
count_summary = (
    count_summary
    .sort_values("전체등장횟수", ascending=False)
    .reset_index(drop=True)
)

# 상위 30개 단어 확인
count_summary.head(30)

,단어,전체등장횟수
0,에디션,12
1,리커버,8
2,기념,8
3,위한,7
4,해커스,7
5,토익,6
6,나는,5
7,내가,5
8,스페셜,5
9,싶은,4


### 실행 결과 요약

- 전체 빈도 확인: CountVectorizer로 추출한 단어 중 전체 도서 제목에서 자주 등장한 상위 30개 단어를 확인했다.
- 최상위 단어 확인: `에디션`이 **12회**로 가장 많이 등장했고, `리커버`와 `기념`이 각각 **8회**로 뒤를 이었다.
- 주요 단어 확인: `해커스`는 **7회**, `토익`은 **6회**, `스페셜`은 **5회** 등장했다.
- 표현 형태 확인: `위한`, `나는`, `내가`, `않는다`처럼 조사나 어미가 붙은 표현도 그대로 단어로 집계되었다.
- 숫자·영문 확인: `2026`, `2025`, `lc`, `rc`, `voca`, `etf로` 같은 숫자와 영문이 포함된 표현도 토큰으로 추출되었다.
- Chapter 02와 차이 확인: Kiwi 형태소 분석과 불용어 처리를 적용했던 이전 결과와 달리, 기본 CountVectorizer는 자체 토큰 기준을 사용하기 때문에 상위 단어와 빈도가 다르게 나타났다.
- 결과 확인: CountVectorizer의 기본 토큰화 결과를 해석할 때는 단순 빈도뿐 아니라 어떤 형태의 단어가 그대로 포함되었는지도 함께 확인해야 함을 알 수 있다.

## 실습 17. Count 상위 단어 저장하기

 

상위 30개 단어를 저장합니다.

In [17]:
# 상위 30개 단어만 선택
count_top30 = count_summary.head(30)

# CSV 파일로 저장
count_top30.to_csv(
    "chapter03_count_top_terms.csv",
    index=False,          # 인덱스는 저장하지 않음
    encoding="utf-8-sig"  # 한글이 깨지지 않도록 저장
)

# 저장한 파일 다시 불러오기
count_top30_check = pd.read_csv(
    "chapter03_count_top_terms.csv",
    encoding="utf-8-sig"
)

# 앞의 5개 행 확인
count_top30_check.head()

,단어,전체등장횟수
0,에디션,12
1,리커버,8
2,기념,8
3,위한,7
4,해커스,7


### 실행 결과 요약

- 파일 재로드 확인: 저장한 `chapter03_count_top_terms.csv` 파일을 다시 불러와 앞의 5개 행을 확인했다.
- 상위 단어 확인: `에디션`이 **12회**로 가장 많이 등장했다.
- 다음 순위 확인: `리커버`와 `기념`은 각각 **8회**, `위한`과 `해커스`는 각각 **7회** 등장했다.
- 데이터 형식 확인: `단어`와 `전체등장횟수` 컬럼이 정상적으로 유지되었다.
- 결과 확인: 저장한 CountVectorizer 상위 단어 결과가 한글 깨짐 없이 정상적으로 다시 불러와졌음을 확인했다.

## 실습 18. Count 방식의 한계 생각해 보기

 

CountVectorizer는 이해하기 쉽지만 한 가지 중요한 문제가 있습니다.

### CountVectorizer의 한계

- 정의: CountVectorizer는 각 단어가 문서에 몇 번 등장했는지를 기준으로 숫자 벡터를 만드는 방식이다.
- 한계: 단어의 등장 횟수가 높다고 해서 반드시 중요한 단어라고 볼 수는 없다.
- 예시: `책`, `도서`, `이야기`, `세상`처럼 여러 문서에서 자주 등장하는 단어는 빈도는 높지만 문서를 구분하는 힘은 약할 수 있다.
- 비교: 반대로 특정 문서에서만 두드러지게 등장하는 단어는 그 문서의 특징을 더 잘 나타낼 수 있다.
- 보완 필요: 단순 등장 횟수뿐 아니라 **전체 문서에서 얼마나 흔한 단어인지**도 함께 고려해야 한다.
- 다음 단계: 이러한 한계를 보완하기 위해 **TF-IDF**를 사용한다.

## 실습 19. TF 이해하기

 

TF는 Term Frequency의 약자입니다.

### TF 이해하기

- 정의: TF는 **Term Frequency**의 약자로, 한 문서 안에서 특정 단어가 얼마나 자주 등장하는지를 나타낸다.
- 예시: `파이썬 파이썬 데이터 분석`에서는 `파이썬`이 2번, `데이터`와 `분석`이 각각 1번 등장한다.
- 의미: 같은 문서 안에서는 더 자주 등장한 단어의 TF가 더 높다고 볼 수 있다.
- 주의점: scikit-learn의 TF-IDF 최종 값에는 TF뿐 아니라 IDF와 정규화가 함께 적용된다.
- 결과 해석: 따라서 최종 TF-IDF 값을 단순한 단어 등장 횟수와 같은 값으로 해석하면 안 된다.

## 실습 20. DF 이해하기

 

DF는 Document Frequency입니다.

### DF 이해하기

- 정의: DF는 **Document Frequency**의 약자로, 특정 단어가 전체 문서 중 몇 개의 문서에 등장하는지를 나타낸다.
- 예시: 문서가 100개 있을 때 `데이터`라는 단어가 80개 문서에 등장한다면 DF가 높은 편이다.
- 비교: `약동학`이라는 단어가 2개 문서에만 등장한다면 DF가 낮은 편이다.
- 의미: DF가 높을수록 해당 단어가 여러 문서에 널리 등장한다는 뜻이다.
- 활용: DF는 이후 IDF를 계산할 때 사용되며, 너무 흔한 단어의 중요도를 낮추는 데 활용된다.

## 실습 21. IDF 이해하기

 

IDF는 Inverse Document Frequency입니다.

### IDF 이해하기

- 정의: IDF는 **Inverse Document Frequency**의 약자로, 특정 단어가 전체 문서에서 얼마나 드문지를 반영하는 값이다.
- 흔한 단어: 거의 모든 문서에 등장하는 단어는 문서를 구분하는 힘이 약할 수 있어 IDF가 상대적으로 작아진다.
- 드문 단어: 적은 수의 문서에만 등장하는 단어는 특정 문서를 구분하는 데 도움이 될 수 있어 IDF가 상대적으로 커질 수 있다.
- 역할: 전체 문서에서 자주 등장하는 단어의 가중치는 낮추고, 상대적으로 드문 단어의 가중치는 높이는 데 사용된다.
- 주의점: scikit-learn은 기본적으로 smoothing이 포함된 IDF 공식을 사용한다.
- 학습 포인트: 개념 단계에서는 공식을 외우기보다 **흔한 단어의 중요도는 낮추고, 드문 단어의 중요도는 높인다**고 이해하면 된다.

## 실습 22. TF-IDF를 한 문장으로 정리하기

 

TF-IDF는 다음 두 관점을 결합합니다.

### TF-IDF 이해하기

- 정의: TF-IDF는 **Term Frequency - Inverse Document Frequency**의 약자로, 한 문서 안에서의 단어 빈도와 전체 문서에서의 희소성을 함께 반영하는 값이다.
- TF 관점: 특정 단어가 현재 문서에서 얼마나 자주 등장하는지를 본다.
- IDF 관점: 해당 단어가 전체 문서에서는 얼마나 흔하거나 드문지를 본다.
- 의미: 현재 문서에서는 자주 등장하지만 전체 문서에서는 너무 흔하지 않은 단어가 상대적으로 높은 TF-IDF 값을 가질 수 있다.
- 해석 주의: TF-IDF가 높다고 해서 현실 세계에서 절대적으로 중요한 단어라는 뜻은 아니다.
- 결과 해석: 현재 문서 집합과 현재 Vectorizer 설정 안에서 상대적으로 문서를 구분하는 데 도움이 되는 특징이라고 이해하면 된다.

## 실습 23. 작은 예제로 TfidfVectorizer 적용하기

In [20]:
# TF-IDF Vectorizer 불러오기
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 객체 생성
tfidf_sample_vectorizer = TfidfVectorizer()

# 예제 문장을 TF-IDF 벡터로 변환
X_tfidf_sample = tfidf_sample_vectorizer.fit_transform(sample_docs)

# 단어 순서 확인
tfidf_sample_terms = tfidf_sample_vectorizer.get_feature_names_out()

# CountVectorizer 결과를 DataFrame으로 변환
count_compare_df = pd.DataFrame(
    X_count_sample.toarray(),
    columns=feature_names,
    index=sample_docs
)

# TF-IDF 결과를 DataFrame으로 변환
tfidf_compare_df = pd.DataFrame(
    X_tfidf_sample.toarray(),
    columns=tfidf_sample_terms,
    index=sample_docs
)

# 기존 Count 0/1 행렬 확인
print("CountVectorizer 결과")
display(count_compare_df)

# TF-IDF 행렬 확인
print("TF-IDF 결과")
display(tfidf_compare_df.round(3))

CountVectorizer 결과


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


TF-IDF 결과


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,0.577,0.000,0.577,0.000,0.577
파이썬 머신러닝,0.000,0.796,0.000,0.000,0.605
데이터 분석 입문,0.518,0.000,0.518,0.681,0.000


### 실행 결과 요약

- Count 결과 확인: CountVectorizer에서는 각 단어의 등장 횟수가 `0` 또는 `1`의 정수 값으로 나타났다.
- TF-IDF 결과 확인: 같은 문장들이 TF-IDF에서는 `0.577`, `0.796`, `0.605`처럼 소수 값으로 변환되었다.
- 반복 단어 확인: `파이썬`, `데이터`, `분석`처럼 여러 문서에 등장하는 단어는 상대적으로 낮은 TF-IDF 값을 가질 수 있었다.
- 드문 단어 확인: `머신러닝`, `입문`처럼 특정 문서에서만 등장하는 단어는 상대적으로 높은 TF-IDF 값을 가졌다.
- 문서별 차이 확인: 같은 단어라도 문서 구성과 정규화 결과에 따라 TF-IDF 값이 달라질 수 있었다.
- 결과 확인: CountVectorizer가 단순 등장 횟수를 나타내는 반면, TF-IDF는 전체 문서에서의 희소성까지 반영해 단어별 가중치를 다르게 계산함을 확인했다.

## 실습 24. TfidfVectorizer가 만든 단어 확인하기

In [21]:
# TF-IDF에서 사용한 단어 확인
tfidf_terms = tfidf_sample_vectorizer.get_feature_names_out()

print("TF-IDF 단어 목록:")
print(tfidf_terms)

TF-IDF 단어 목록:
['데이터' '머신러닝' '분석' '입문' '파이썬']


### 실행 결과 요약

- 단어 목록 확인: TfidfVectorizer가 `데이터`, `머신러닝`, `분석`, `입문`, `파이썬` 총 **5개 단어**를 사용한 것을 확인했다.
- 열 순서 확인: TF-IDF 행렬의 열은 `데이터`, `머신러닝`, `분석`, `입문`, `파이썬` 순서로 구성되었다.
- Count와 비교: 앞에서 확인한 CountVectorizer의 단어 순서와 동일하게 나타났다.
- 구조 확인: 각 행은 문서를 의미하고, 각 열은 단어를 의미하며, 각 값은 해당 단어의 TF-IDF 가중치를 나타낸다.
- 결과 확인: TF-IDF 역시 문서를 단어 기준의 숫자 벡터 형태로 표현한다는 것을 확인했다.

## 실습 25. 단어별 IDF 값 확인하기

 

TfidfVectorizer가 학습한 IDF 값을 직접 확인할 수 있습니다.

In [22]:
# 단어별 IDF 값을 DataFrame으로 정리
idf_df = pd.DataFrame({
    "단어": tfidf_sample_vectorizer.get_feature_names_out(),
    "IDF": tfidf_sample_vectorizer.idf_
})

# IDF가 높은 순서대로 정렬해서 확인
idf_df = (
    idf_df
    .sort_values("IDF", ascending=False)
    .reset_index(drop=True)
)

idf_df

,단어,IDF
0,머신러닝,1.693147
1,입문,1.693147
2,데이터,1.287682
3,분석,1.287682
4,파이썬,1.287682


### 실행 결과 요약

- IDF 값 확인: `머신러닝`과 `입문`의 IDF가 각각 **1.693147**로 가장 높게 나타났다.
- 공통 단어 확인: `데이터`, `분석`, `파이썬`의 IDF는 각각 **1.287682**로 더 낮게 나타났다.
- 문서 분포 비교: `머신러닝`과 `입문`은 일부 문서에만 등장해 상대적으로 드문 단어로 평가되었다.
- 반복 단어 비교: `데이터`, `분석`, `파이썬`은 여러 문서에 반복해서 등장해 상대적으로 IDF가 낮아졌다.
- 결과 확인: 전체 문서에서 드물게 등장하는 단어일수록 IDF가 높고, 여러 문서에 널리 등장하는 단어일수록 IDF가 낮아짐을 확인했다.

## 실습 26. 실제 도서 제목을 TF-IDF로 변환하기

 

이제 실제 데이터에 적용합니다.

In [23]:
# 실제 도서 제목에 TF-IDF 적용
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 객체 생성
tfidf_vectorizer = TfidfVectorizer()

# 제목을 TF-IDF 벡터로 변환
X_tfidf = tfidf_vectorizer.fit_transform(titles)

# TF-IDF에서 사용한 단어 목록
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

# TF-IDF 행렬 크기 확인
print("TF-IDF 행렬 크기:", X_tfidf.shape)
print("문서 수:", X_tfidf.shape[0])
print("단어 수:", X_tfidf.shape[1])

# CountVectorizer 결과와 크기 비교
print("\nCount shape:", X_count.shape)
print("TF-IDF shape:", X_tfidf.shape)

TF-IDF 행렬 크기: (199, 536)
문서 수: 199
단어 수: 536

Count shape: (199, 536)
TF-IDF shape: (199, 536)


### 실행 결과 요약

- TF-IDF 행렬 크기 확인: 실제 도서 제목을 TF-IDF로 변환한 결과 행렬 크기는 **(199, 536)**으로 나타났다.
- 문서 수 확인: TF-IDF에 사용된 도서 제목은 총 **199개**이다.
- 단어 수 확인: TfidfVectorizer가 만든 단어는 총 **536개**이다.
- Count 비교: CountVectorizer의 행렬 크기도 **(199, 536)**으로 TF-IDF와 동일하게 나타났다.
- 구조 비교: 두 방식 모두 같은 제목 데이터와 같은 기본 토큰 기준을 사용했기 때문에 행과 열의 개수가 동일했다.
- 결과 확인: CountVectorizer와 TF-IDF는 같은 문서와 단어 구조를 사용하지만, Count는 등장 횟수를 나타내고 TF-IDF는 단어별 가중치를 나타낸다는 차이가 있다.

## 실습 27. 첫 번째 도서의 TF-IDF 주요 단어 확인하기

 

첫 번째 제목을 다시 확인합니다.

In [24]:
# 첫 번째 도서 제목 확인
print("도서 제목:", titles.iloc[0])

# 첫 번째 도서의 TF-IDF 행 가져오기
first_tfidf_row = X_tfidf.getrow(0)

# 0보다 큰 TF-IDF 값을 가진 단어만 DataFrame으로 정리
first_tfidf_df = pd.DataFrame({
    "단어": tfidf_terms[first_tfidf_row.indices],
    "TF-IDF": first_tfidf_row.data
})

# TF-IDF가 높은 순서대로 정렬
first_tfidf_df = (
    first_tfidf_df
    .sort_values("TF-IDF", ascending=False)
    .reset_index(drop=True)
)

# 결과 확인
first_tfidf_df

도서 제목: 소년이 온다


,단어,TF-IDF
0,소년이,0.707107
1,온다,0.707107


### 실행 결과 요약

- 첫 번째 도서 확인: 첫 번째 도서 제목은 `소년이 온다`로 확인되었다.
- 단어 추출 확인: TF-IDF 결과에서 `소년이`, `온다` 두 단어가 추출되었다.
- TF-IDF 값 확인: 두 단어의 TF-IDF 값은 각각 **0.707107**로 동일하게 나타났다.
- 값 비교: 두 단어가 이 제목에서 각각 한 번씩 등장하고, 전체 문서에서의 희소성도 비슷하게 반영되어 같은 TF-IDF 값을 가졌다.
- 결과 확인: 첫 번째 도서에서는 `소년이`와 `온다`가 동일한 비중의 특징으로 표현되었음을 확인했다.

## 실습 28. 여러 도서의 주요 TF-IDF 단어 확인 함수 만들기

 

반복해서 확인하기 위해 작은 함수를 만들 수 있습니다.

In [25]:
# 여러 도서의 주요 TF-IDF 단어를 확인하는 함수
def show_top_tfidf_terms(doc_index, top_n=5):
    
    # 선택한 도서의 TF-IDF 행 가져오기
    row = X_tfidf.getrow(doc_index)

    # 단어와 TF-IDF 값을 DataFrame으로 정리
    result = pd.DataFrame({
        "단어": tfidf_terms[row.indices],
        "TF-IDF": row.data
    })

    # TF-IDF가 높은 순서대로 정렬 후 상위 top_n개 선택
    result = (
        result
        .sort_values("TF-IDF", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # 도서 제목 출력
    print("도서 제목:", titles.iloc[doc_index])

    return result


# 여러 도서 확인
display(show_top_tfidf_terms(0, top_n=5))
display(show_top_tfidf_terms(10, top_n=5))
display(show_top_tfidf_terms(100, top_n=5))

도서 제목: 소년이 온다


,단어,TF-IDF
0,소년이,0.707107
1,온다,0.707107


도서 제목: 작별하지 않는다


,단어,TF-IDF
0,작별하지,0.752078
1,않는다,0.659074


도서 제목: 말투만 바꿨을 뿐인데


,단어,TF-IDF
0,말투만,0.57735
1,바꿨을,0.57735
2,뿐인데,0.57735


### 실행 결과 요약

- 0번 도서 확인: `소년이 온다`에서는 `소년이`, `온다`가 각각 **0.707107**로 같은 TF-IDF 값을 가졌다.
- 10번 도서 확인: `작별하지 않는다`에서는 `작별하지`가 **0.752078**, `않는다`가 **0.659074**로 나타났다.
- 100번 도서 확인: `말투만 바꿨을 뿐인데`에서는 `말투만`, `바꿨을`, `뿐인데`가 각각 **0.57735**로 동일하게 나타났다.
- 단어별 차이 확인: 같은 제목 안에서도 전체 문서에서의 희소성 차이에 따라 TF-IDF 값이 서로 다르게 나타날 수 있었다.
- 동일 가중치 확인: 문서 내 등장 횟수와 전체 문서에서의 분포가 비슷한 단어들은 동일한 TF-IDF 값을 가질 수 있었다.
- 결과 확인: `show_top_tfidf_terms()` 함수를 사용해 여러 도서의 주요 단어와 TF-IDF 값을 반복해서 비교할 수 있음을 확인했다.

## 실습 29. 전체 데이터에서 평균 TF-IDF가 높은 단어 확인하기

 

각 단어의 TF-IDF 값을 전체 문서에서 평균내어 탐색할 수 있습니다.

In [26]:
# 각 단어의 평균 TF-IDF 계산
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

# 단어와 평균 TF-IDF 값을 DataFrame으로 정리
tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "평균_TFIDF": mean_tfidf
})

# 평균 TF-IDF가 높은 순서대로 정렬
tfidf_summary = (
    tfidf_summary
    .sort_values("평균_TFIDF", ascending=False)
    .reset_index(drop=True)
)

# 상위 30개 단어 확인
tfidf_summary.head(30)

,단어,평균_TFIDF
0,에디션,0.019330
1,리커버,0.013861
2,기념,0.012711
3,위한,0.012087
4,해커스,0.012072
5,토익,0.010950
6,스페셜,0.009901
7,나는,0.009786
8,내가,0.009454
9,대하여,0.008822


### 실행 결과 요약

- 평균 TF-IDF 확인: 전체 도서 제목에서 단어별 평균 TF-IDF 값을 계산해 상위 30개 단어를 확인했다.
- 최상위 단어 확인: `에디션`이 **0.019330**으로 가장 높은 평균 TF-IDF 값을 가졌다.
- 다음 순위 확인: `리커버`는 **0.013861**, `기념`은 **0.012711**, `위한`은 **0.012087**, `해커스`는 **0.012072**로 나타났다.
- 주요 단어 확인: `토익`, `트렌드`, `필사`, `일본어`, `voca` 등 주제나 학습 분야와 관련된 단어도 상위에 포함되었다.
- 표현 형태 확인: `위한`, `나는`, `내가`, `않는다`, `돈의`처럼 조사나 어미가 붙은 형태도 그대로 포함되었다.
- 숫자·영문 확인: `2025`, `2026`, `lc`, `rc`, `voca`, `30만` 같은 숫자와 영문 표현도 TF-IDF 특징으로 사용되었다.
- 결과 확인: 평균 TF-IDF 상위 단어는 현재 문서 집합에서 상대적으로 큰 가중치를 가진 텍스트 특징이며, 이를 베스트셀러의 원인으로 해석해서는 안 된다.

## 실습 30. TF-IDF 상위 단어 저장하기

In [27]:
# 평균 TF-IDF 상위 30개 단어 선택
tfidf_top30 = tfidf_summary.head(30)

# CSV 파일로 저장
tfidf_top30.to_csv(
    "chapter03_tfidf_top_terms.csv",
    index=False,          # 인덱스는 저장하지 않음
    encoding="utf-8-sig"  # 한글 깨짐 방지
)

# 저장한 파일 다시 불러오기
tfidf_top30_check = pd.read_csv(
    "chapter03_tfidf_top_terms.csv",
    encoding="utf-8-sig"
)

# 앞의 5개 행 확인
tfidf_top30_check.head()

,단어,평균_TFIDF
0,에디션,0.019330
1,리커버,0.013861
2,기념,0.012711
3,위한,0.012087
4,해커스,0.012072


### 실행 결과 요약

- 파일 재로드 확인: 저장한 `chapter03_tfidf_top_terms.csv` 파일을 다시 불러와 앞의 5개 행을 확인했다.
- 최상위 단어 확인: `에디션`이 평균 TF-IDF **0.019330**으로 가장 높게 나타났다.
- 다음 순위 확인: `리커버`는 **0.013861**, `기념`은 **0.012711**, `위한`은 **0.012087**, `해커스`는 **0.012072**로 나타났다.
- 데이터 형식 확인: `단어`와 `평균_TFIDF` 컬럼이 정상적으로 유지되었다.
- 결과 확인: 평균 TF-IDF 상위 단어 결과가 한글 깨짐 없이 정상적으로 저장되고 다시 불러와졌음을 확인했다.

## 실습31. Count 상위 단어와 TF-IDF 상위 단어 비교하기

In [28]:
# Count 상위 단어와 TF-IDF 상위 단어를 나란히 비교
comparison = pd.DataFrame({
    "Count_상위단어": count_top30["단어"].reset_index(drop=True),
    "Count_전체등장횟수": count_top30["전체등장횟수"].reset_index(drop=True),
    "TFIDF_상위단어": tfidf_top30["단어"].reset_index(drop=True),
    "평균_TFIDF": tfidf_top30["평균_TFIDF"].reset_index(drop=True)
})

# 앞의 20개 결과 확인
comparison.head(20)

,Count_상위단어,Count_전체등장횟수,TFIDF_상위단어,평균_TFIDF
0,에디션,12,에디션,0.019330
1,리커버,8,리커버,0.013861
2,기념,8,기념,0.012711
3,위한,7,위한,0.012087
4,해커스,7,해커스,0.012072
5,토익,6,토익,0.010950
6,나는,5,스페셜,0.009901
7,내가,5,나는,0.009786
8,스페셜,5,내가,0.009454
9,싶은,4,대하여,0.008822


### 실행 결과 요약

- 상위 순위 비교: `에디션`, `리커버`, `기념`, `위한`, `해커스`, `토익`은 Count와 평균 TF-IDF 모두 상위에 나타났다.
- 순위 변화 확인: `스페셜`은 Count에서는 9위였지만 평균 TF-IDF에서는 7위로 올라갔다.
- 상대적 하락 확인: `나는`, `내가`, `싶은`, `rc`, `lc` 등은 Count 순위와 TF-IDF 순위가 서로 다르게 나타났다.
- 새롭게 눈에 띈 단어: `흔한남매`, `기술`은 Count 상위 20개에는 없었지만 평균 TF-IDF 상위 20개에는 포함되었다.
- 공통 특징 확인: 두 방식 모두 단순히 전혀 다른 결과를 만드는 것이 아니라, 자주 등장하는 단어 중 일부는 공통적으로 상위에 유지되었다.
- 차이 해석: Count는 전체 등장 횟수를 중심으로 보지만, TF-IDF는 각 단어가 여러 문서에 얼마나 널리 퍼져 있는지도 함께 반영하기 때문에 순위 차이가 발생했다.
- 결과 확인: 같은 단어라도 단순 빈도와 문서 전체에서의 희소성을 어떻게 반영하느냐에 따라 중요도가 다르게 평가될 수 있음을 확인했다.

## 실습32. 특정 단어가 몇 개 문서에 등장하는지 확인하기

In [29]:
# 특정 단어가 몇 개의 도서 제목에 등장하는지 확인하는 함수
def document_frequency(term):
    
    # 단어가 vocabulary에 없으면 0 반환
    if term not in count_vectorizer.vocabulary_:
        return 0

    # 해당 단어의 열 번호 확인
    column_index = count_vectorizer.vocabulary_[term]

    # 해당 단어의 열만 선택
    column = X_count[:, column_index]

    # 0보다 큰 값의 개수 = 해당 단어가 등장한 문서 수
    return int((column > 0).sum())


# 예시 단어의 DF 확인
print("데이터 DF:", document_frequency("데이터"))
print("파이썬 DF:", document_frequency("파이썬"))
print("에디션 DF:", document_frequency("에디션"))
print("토익 DF:", document_frequency("토익"))

데이터 DF: 0
파이썬 DF: 0
에디션 DF: 12
토익 DF: 6


### 실행 결과 요약

- DF 확인: `에디션`은 **12개 문서**, `토익`은 **6개 문서**에서 등장했다.
- 미등장 단어 확인: `데이터`와 `파이썬`은 실제 도서 제목의 CountVectorizer 단어 사전에 존재하지 않아 DF가 **0**으로 나타났다.
- 의미 확인: DF가 클수록 해당 단어가 더 많은 도서 제목에 등장하고, DF가 작을수록 일부 제목에만 등장한다.
- 빈도 비교: 이번 데이터에서는 `에디션`이 `토익`보다 더 많은 제목에 널리 등장한 것을 확인했다.
- 주의점: `데이터`와 `파이썬`의 DF가 0인 것은 Vectorizer가 단어를 인식하지 못했다는 뜻이 아니라, **현재 199개 실제 도서 제목에 해당 토큰이 없다는 뜻**이다.
- 결과 확인: 단어의 전체 등장 횟수뿐 아니라 실제 몇 개의 문서에 등장했는지를 DF로 직접 확인할 수 있었다.

## 실습33. CountVectorizer와 TfidfVectorizer의 역할 비교

### CountVectorizer와 TfidfVectorizer 비교

| 구분 | CountVectorizer | TfidfVectorizer |
| --- | --- | --- |
| 기본 값 | 단어 등장 횟수 | TF-IDF 가중치 |
| 값 형태 | 주로 정수 | 실수 |
| 전체 문서의 흔함 고려 | 직접 고려하지 않음 | IDF로 고려 |
| 장점 | 단순하고 직관적 | 흔한 단어의 영향 일부 조정 |
| 주요 활용 | 빈도 기반 특징 | 분류, 유사도 등 텍스트 특징 |

- 해석: CountVectorizer는 단어가 몇 번 등장했는지를 그대로 반영하고, TfidfVectorizer는 단어의 등장 빈도와 전체 문서에서의 희소성을 함께 반영한다.
- 주의점: 둘 중 하나가 항상 더 좋은 것은 아니며, 분석 목적과 데이터에 따라 실제 성능을 비교해 선택해야 한다.

## 실습 47. 일부 도서 결과를 직접 검증하기

In [30]:
# 직접 확인할 도서 번호
sample_indices = [0, 10, 20]

# 선택한 도서들의 주요 TF-IDF 단어 확인
for index in sample_indices:
    
    # 데이터 범위를 벗어나지 않을 때만 실행
    if index < len(titles):
        print("=" * 60)
        print("도서 제목:", titles.iloc[index])
        
        # TF-IDF 상위 5개 단어 확인
        display(show_top_tfidf_terms(index, top_n=5))

도서 제목: 소년이 온다
도서 제목: 소년이 온다


,단어,TF-IDF
0,소년이,0.707107
1,온다,0.707107


도서 제목: 작별하지 않는다
도서 제목: 작별하지 않는다


,단어,TF-IDF
0,작별하지,0.752078
1,않는다,0.659074


도서 제목: 쇼펜하우어 인생수업(30만 부 기념 개정증보판)
도서 제목: 쇼펜하우어 인생수업(30만 부 기념 개정증보판)


,단어,TF-IDF
0,개정증보판,0.498481
1,쇼펜하우어,0.462422
2,인생수업,0.462422
3,30만,0.436838
4,기념,0.364720


### 실행 결과 요약

- 0번 도서 확인: `소년이 온다`에서는 `소년이`, `온다`가 각각 **0.707107**로 나타났으며 실제 제목에 포함된 단어와 일치했다.
- 10번 도서 확인: `작별하지 않는다`에서는 `작별하지`가 **0.752078**, `않는다`가 **0.659074**로 나타났고 두 단어 모두 실제 제목에 포함되어 있었다.
- 20번 도서 확인: `쇼펜하우어 인생수업(30만 부 기념 개정증보판)`에서는 `개정증보판`이 **0.498481**로 가장 높았고, `쇼펜하우어`, `인생수업`, `30만`, `기념`이 뒤를 이었다.
- 숫자 표현 확인: `30만`처럼 숫자가 포함된 표현도 하나의 feature로 추출되었다.
- 일반 표현 확인: `기념`, `개정증보판`처럼 책의 내용보다는 판형이나 마케팅 성격에 가까운 단어도 높은 TF-IDF 값을 가질 수 있었다.
- 형태소 분석 필요성 확인: 기본 TfidfVectorizer는 형태소 분석이나 불용어 처리를 하지 않기 때문에, 분석 목적과 직접 관련이 적은 표현도 특징으로 남을 수 있음을 확인했다.
- 결과 확인: 상위 TF-IDF 단어들은 실제 제목에 존재했지만, 분석 목적에 따라 형태소 분석과 불용어 처리를 추가하면 더 의미 있는 특징을 얻을 수 있음을 확인했다.

## 실습 50. 왜 Chapter 04에서는 전체 데이터에 먼저 fit하면 안 될까?

 

이번 Chapter에서는 Vectorizer 자체를 이해하기 위해 전체 제목을 사용했습니다.

### 데이터 누수(Data Leakage) 이해하기

- 정의: 데이터 누수는 모델을 평가할 때 사용해야 하는 테스트 데이터의 정보가 학습 과정에 미리 들어가는 문제이다.
- 이번 Chapter: Vectorizer의 동작 원리를 이해하기 위한 탐색 목적이므로 전체 도서 제목에 `fit_transform()`을 적용했다.
- 문제 상황: 전체 데이터에 먼저 `TfidfVectorizer.fit_transform()`을 적용한 뒤 train/test로 나누면, Vectorizer가 단어 사전과 IDF를 만들 때 테스트 데이터 정보까지 사용하게 된다.
- 왜 문제인가: 테스트 데이터는 원래 학습 과정에서 보지 않은 데이터여야 하므로, 이 정보가 미리 반영되면 모델 성능 평가가 실제보다 좋게 보일 수 있다.
- 올바른 방향: 모델 평가 단계에서는 먼저 train/test를 나눈 뒤, Vectorizer는 train 데이터에만 `fit()`하고 test 데이터에는 `transform()`만 적용해야 한다.
- 결과 확인: Chapter 03에서는 개념 학습을 위해 전체 데이터에 fit했지만, Chapter 04의 모델 평가에서는 데이터 누수를 막기 위해 학습 데이터와 테스트 데이터를 분리한 뒤 Vectorizer를 적용해야 한다.

In [31]:
# 데이터 누수 비교 실험

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 제목 데이터를 train / test로 먼저 분리
X_train, X_test = train_test_split(
    titles,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# 1. 올바른 방식
# train 데이터에만 fit
# -----------------------------
correct_vectorizer = TfidfVectorizer()

X_train_correct = correct_vectorizer.fit_transform(X_train)
X_test_correct = correct_vectorizer.transform(X_test)

print("올바른 방식")
print("train 문서 수:", X_train_correct.shape[0])
print("test 문서 수:", X_test_correct.shape[0])
print("학습된 단어 수:", X_train_correct.shape[1])


# -----------------------------
# 2. 잘못된 방식
# 전체 데이터에 먼저 fit한 뒤 분리
# -----------------------------
leak_vectorizer = TfidfVectorizer()

X_all_leak = leak_vectorizer.fit_transform(titles)

print("\n잘못된 방식")
print("전체 문서 수:", X_all_leak.shape[0])
print("학습된 단어 수:", X_all_leak.shape[1])


# -----------------------------
# 3. 두 방식의 단어 사전 비교
# -----------------------------
correct_terms = set(correct_vectorizer.get_feature_names_out())
leak_terms = set(leak_vectorizer.get_feature_names_out())

test_information_terms = leak_terms - correct_terms

print("\n비교 결과")
print("train에만 fit한 단어 수:", len(correct_terms))
print("전체 데이터에 fit한 단어 수:", len(leak_terms))
print("전체 데이터에서만 추가된 단어 수:", len(test_information_terms))

# 테스트 데이터 정보 때문에 추가된 단어 일부 확인
print("\n전체 데이터에 fit했을 때만 포함된 단어 예시:")
print(list(test_information_terms)[:20])

올바른 방식
train 문서 수: 159
test 문서 수: 40
학습된 단어 수: 435

잘못된 방식
전체 문서 수: 199
학습된 단어 수: 536

비교 결과
train에만 fit한 단어 수: 435
전체 데이터에 fit한 단어 수: 536
전체 데이터에서만 추가된 단어 수: 101

전체 데이터에 fit했을 때만 포함된 단어 예시:
['주술회전', '에그박사', '기차역', '앞의', '활판인쇄', '웨이', '경량문명의', '너에게', '여름', '3개의', '29', '인생은', '바뀌지', '부에', '1차', '보듯', '메트로폴리탄', '초호화', '인간', '월가의']


### 실행 결과 요약

- 데이터 분리 확인: 전체 199개 도서 제목을 train **159개**, test **40개**로 나누었다.
- 올바른 방식 확인: train 데이터에만 TF-IDF를 fit한 결과 단어 사전은 **435개**로 생성되었다.
- 잘못된 방식 확인: 전체 데이터 199개에 먼저 TF-IDF를 fit한 결과 단어 사전은 **536개**로 생성되었다.
- 단어 수 차이 확인: 전체 데이터에 fit했을 때 train에만 fit한 경우보다 **101개 단어**가 추가로 학습되었다.
- 누수 예시 확인: `주술회전`, `에그박사`, `기차역`, `활판인쇄`, `여름`, `메트로폴리탄` 등 train 데이터만으로는 알 수 없었던 단어가 전체 데이터 fit 과정에서 미리 포함되었다.
- 데이터 누수 확인: 전체 데이터에 먼저 fit하면 test 데이터의 단어 정보가 단어 사전과 IDF 계산에 반영될 수 있음을 직접 확인했다.
- 결과 확인: 모델 평가에서는 반드시 **train 데이터에만 fit**하고, test 데이터에는 **transform만 적용**해야 데이터 누수를 막을 수 있다.

## 51.실습 51. Chapter 04에서 사용할 올바른 순서 미리 보기

 

다음 Chapter에서는 다음 순서를 사용합니다.

### Chapter 04에서 사용할 올바른 순서

원본 데이터  
↓  
`X`와 `y` 준비  
↓  
train / test 분리  
↓  
TF-IDF를 train 데이터에 `fit`  
↓  
train 데이터 `transform`  
↓  
test 데이터는 `transform`만 수행  
↓  
Naive Bayes 모델 학습  
↓  
test 데이터 예측  
↓  
모델 성능 평가

- train 학습: `TfidfVectorizer`는 **train 데이터에만 `fit()`**하여 단어 사전과 IDF를 학습한다.
- train 변환: train 데이터는 `fit_transform()`으로 TF-IDF 벡터로 변환한다.
- test 변환: test 데이터는 이미 학습된 Vectorizer를 사용해 `transform()`만 수행한다.
- 핵심 원칙: test 데이터에는 `fit_transform()`을 사용하지 않는다.

```python
# 올바른 방식
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)